In [ ]:
#!pip install xformers peft accelerate bitsandbytes datasets flash-attn lightning wandb
# for colab
#!pip install xformers peft accelerate bitsandbytes datasets lightning wandb
#!pip install "trl<0.12.0" # when use train_rl_model
#!pip install trl # when use train_reward_model

#!pip uninstall -y trl transformers
#!pip install -q "transformers==4.41.2" "trl==0.9.4" accelerate datasets peft bitsandbytes wandb

In [12]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [13]:
!zip -r reward_model.zip /content/reward_model

  adding: content/reward_model/ (stored 0%)
  adding: content/reward_model/tokenizer_config.json (deflated 75%)
  adding: content/reward_model/training_args.bin (deflated 53%)
  adding: content/reward_model/vocab.json (deflated 74%)
  adding: content/reward_model/config.json (deflated 51%)
  adding: content/reward_model/special_tokens_map.json (deflated 77%)
  adding: content/reward_model/model.safetensors (deflated 14%)
  adding: content/reward_model/merges.txt (deflated 76%)
  adding: content/reward_model/tokenizer.json (deflated 80%)


In [ ]:
!mkdir -p /content/drive/MyDrive/ru_text_simplification

In [14]:
!cp /content/reward_model.zip /content/drive/MyDrive/ru_text_simplification/

In [1]:
!pip install -q \
  "transformers==4.41.2" \
  "trl==0.9.4" \
  "peft==0.10.0" \
  "accelerate==0.31.0" \
  "datasets==2.20.0" \
  bitsandbytes wandb

In [2]:
!pip install pymorphy3 pymorphy3-dicts-ru

In [ ]:
import transformers, trl, peft, accelerate, datasets
print("transformers", transformers.__version__)
print("trl", trl.__version__)
print("peft", peft.__version__)
print("accelerate", accelerate.__version__)
print("datasets", datasets.__version__)

transformers 4.41.2
trl 0.9.4
peft 0.10.0
accelerate 0.31.0
datasets 2.20.0


# data_utils

In [ ]:
import pandas as pd
#import json
#import requests
from typing import Dict, List, Optional, Tuple, Union
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split

# new
# helper for cefr conversion
def bucket(lbl: str) -> str:
    if lbl in ("A1", "A2"):
        return "A"
    if lbl in ("B1", "B2"):
        return "B"
    if lbl in ("C1", "C2"):
        return "C"
    raise ValueError(f"Unknown CEFR label: {lbl}")

# new
# helper for cefr conversion
def split_ABC(ts, ls):
    A, B, C = [], [], []
    for t, l in zip(ts, ls):
        b = bucket(l)
        if b == "A":
            A.append(t)
        elif b == "B":
            B.append(t)
        else:
            C.append(t)
    return A, B, C

# unchanged
def get_preference_pairs(priority_list: List[str],
                        other_list_1: List[str],
                        other_list_2: List[str]) -> Tuple[List[str], List[str]]:
    """Create preference pairs for reward modeling."""
    chosen = []
    rejected = []

    # Add samples from priority list as chosen samples
    for sample in priority_list:
        chosen.append(sample)

    # Add samples from other lists as rejected samples
    for sample in other_list_1:
        rejected.append(sample)
    for sample in other_list_2:
        rejected.append(sample)

    # Ensure equal number of chosen/rejected pairs
    rejected = shuffle(rejected, random_state=0)
    if len(chosen) < len(rejected):
        rejected = rejected[:len(chosen)]
    else:
        rejected = [rejected[i % len(rejected)] for i in range(len(chosen))]

    return chosen, rejected


# changed
def load_cefr_data(level: str, mode: str = "reward") -> Union[Dict[str, Dict[str, List[str]]], List[str]]:
    """Load and prepare CEFR data for training.

    Args:
        level: Target CEFR level (A, B, or C)
        mode: Either "reward" for reward modeling or "rl" for RL training

    Returns:
        For reward mode: Dict with train/eval data containing chosen/rejected pairs
        For RL mode: List of training sentences
    """
    # changed
    # load CEFR sentences
    df = pd.read_csv("CEFR_level_sentences.csv", sep=";", encoding="cp1251")
    df = df[["fragment", "textbook-assigned cefr level"]].dropna()
    df["fragment"] = df["fragment"].astype(str).str.strip()
    df["textbook-assigned cefr level"] = df["textbook-assigned cefr level"].astype(str).str.strip().str.upper()
    df = df[df["fragment"] != ""]

    texts = df["fragment"].tolist()
    labels = df["textbook-assigned cefr level"].tolist()

    # changed
    # train dev test split 80/10/10
    try:
        train_texts, temp_texts, train_labels, temp_labels = train_test_split(
            texts, labels, test_size=0.2, random_state=0, stratify=labels
        )
        dev_texts, eval_texts, dev_labels, eval_labels = train_test_split(
            temp_texts, temp_labels, test_size=0.5, random_state=0, stratify=temp_labels
        )
    except Exception:
        train_texts, temp_texts, train_labels, temp_labels = train_test_split(
            texts, labels, test_size=0.2, random_state=0
        )
        dev_texts, eval_texts, dev_labels, eval_labels = train_test_split(
            temp_texts, temp_labels, test_size=0.5, random_state=0
        )

    # changed
    train_A, train_B, train_C = split_ABC(train_texts, train_labels)
    dev_A, dev_B, dev_C = split_ABC(dev_texts, dev_labels)
    test_A, test_B, test_C = split_ABC(eval_texts, eval_labels)


    if mode == "reward":
        # Prepare preference pairs based on target level
        if level == "A":
            train_chosen, train_rejected = get_preference_pairs(train_A, train_B, train_C)
            dev_chosen, dev_rejected = get_preference_pairs(dev_A, dev_B, dev_C)
            eval_chosen, eval_rejected = get_preference_pairs(test_A, test_B, test_C)
        elif level == "B":
            train_chosen, train_rejected = get_preference_pairs(train_B, train_C, train_A)
            dev_chosen, dev_rejected = get_preference_pairs(dev_B, dev_C, dev_A)
            eval_chosen, eval_rejected = get_preference_pairs(test_B, test_C, test_A)
        else:  # level C
            train_chosen, train_rejected = get_preference_pairs(train_C, train_B, train_A)
            dev_chosen, dev_rejected = get_preference_pairs(dev_C, dev_B, dev_A)
            eval_chosen, eval_rejected = get_preference_pairs(test_C, test_B, test_A)

        return {
            "train": {"chosen": train_chosen, "rejected": train_rejected},
            "dev": {"chosen": dev_chosen, "rejected": dev_rejected},
            "eval": {"chosen": eval_chosen, "rejected": eval_rejected}
        }

    elif mode == "rl":  # mode == "rl"
        # For RL training, return sentences of target level
        if level == "A":
            return train_A
        elif level == "B":
            return train_B
        else:
            return train_C
    else:
        raise ValueError(f"Invalid mode: {mode}")


# changed all
def read_complicated_lines(path: str,
                          *,
                          min_len: int = 1,
                          max_cos_sim: Optional[float] = None) -> List[str]:
    """
    Reads csv with source and target sentences

    Returns:
        List[str] of complex source sentences (RL inputs)

    Args:
        path: path to source_target_sentences.csv
        min_len: drop sources shorter than this many characters
        max_cos_sim: keep only rows with cos_sim <= max_cos_sim
                     (to avoid identical source/target)
    """
    df = pd.read_csv(path, sep=";", encoding="cp1251")

    if "source" not in df.columns:
        raise ValueError(f"Expected column 'source' in {path}, got columns: {list(df.columns)}")

    src = df["source"].dropna().astype(str).str.strip()
    src = src[src != ""]

    # optional quality filters
    if min_len > 1:
        src = src[src.str.len() >= min_len]

    if max_cos_sim is not None:
        if "cos_sim" not in df.columns:
            raise ValueError("max_cos_sim was set but column 'cos_sim' is missing in the CSV.")
        cos = pd.to_numeric(df.loc[src.index, "cos_sim"], errors="coerce")
        src = src[cos <= max_cos_sim]

    return src.tolist()

# changed all
def get_complicated_sentence(path: str,
                             *,
                             min_len: int = 1,
                             max_cos_sim: Optional[float] = None) -> List[str]:
    """
    Returns the same thing as read_complicated_lines().
    """
    return read_complicated_lines(path, min_len=min_len, max_cos_sim=max_cos_sim)

# Modeling

### Lexical reward model

In [4]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import string
from typing import List, Dict, Union
import pandas as pd
import re
from tqdm import tqdm
from typing import Tuple

# new
import nltk
nltk.download("stopwords")
nltk.download('punkt')
from dataclasses import dataclass
from typing import List, Tuple, Union, Dict, Optional
import pymorphy3

# new
CEFR2ABC = {"A1":"A","A2":"A","B1":"B","B2":"B","C1":"C","C2":"C"}

# new
RU_STOP = set(stopwords.words("russian"))


class LexicalRewardModelRU:
    # changed lemmatizer to Russian
    def __init__(self, level: str, word_list_path: str):
        """Initialize lexical reward model for CEFR level."""
        self.level = level
        self.lemmatizer = pymorphy3.MorphAnalyzer()
        self.word_list, self.negative_word_list = self._load_word_lists(word_list_path)
        self.word_reward_dict = {idx: 1.0 for idx in range(len(self.word_list))}
        self.word_freq_dict = {idx: 0 for idx in range(len(self.word_list))}


    # new
    def _lemmatize(self, token: str) -> str:
        return self.lemmatizer.parse(token.lower())[0].normal_form


    # almost not changed, just type check
    def _load_word_lists(self, path: str) -> Tuple[List[Union[str, List[str]]], List[Union[str, List[str]]]]:
        """Load and process CEFR word lists."""
        df = pd.read_csv(path)

        # new
        # safety checks
        df["level"] = df["level"].astype(str).str.strip().str.upper()
        df["base"] = df["base"].astype(str).str.strip()

        # Filter words based on level
        if self.level == "A":
            level_df = df[df['level'].isin(['A1', 'A2'])]
            negative_df = df[df['level'].isin(['B1', 'B2', 'C1', 'C2'])]
        elif self.level == "B":
            level_df = df[df['level'].isin(['B1', 'B2'])]
            negative_df = df[df['level'].isin(['C1', 'C2'])]
        else:  # level C
            level_df = df[df['level'].isin(['C1', 'C2'])]
            negative_df = pd.DataFrame(columns=df.columns)

        word_list = list(set([tuple(self._process_word(w)) for w in level_df['base'].tolist()]))
        negative_word_list = list(set([tuple(self._process_word(w)) for w in negative_df['base'].tolist()]))

        return word_list, negative_word_list


    # unchanged
    def _process_word(self, word: str) -> Union[str, List[str]]:
        """Process a word or phrase from the CEFR profile word list based on a set of rules."""
        # TODO: update to add more rules
        word = word.strip()
        if len(word.split()) > 1:  # phrase
            # 1. remove ", etc.", "be"
            phrase = word.replace(", etc.", "").replace("be", "").replace(" to do sth", "")
            # 2. remove anything in ()
            phrase = re.sub(r'\(.*?\)', '', phrase)
            # 3. split with " or "
            phrases = phrase.split(" or ")[:1]
            processed = []
            for p in phrases:
                # 4. deal with sb/sth
                p = p.replace("sb/sth", "...").replace("sb", "...").replace("sth", "...")
                # 5. deal with / and ()
                if "/" in p:
                    parts = p.split()
                    new_parts = []
                    for part in parts:
                        if "/" in part:
                            part = part.split("/")[0]
                        new_parts.append(part)
                    p = " ".join(new_parts)

                # 6. deal with ...
                if "..." in p:
                    p = [x.lstrip().rstrip() for x in p.split("...") if x.strip()]
                else:
                    p = [p.lstrip().rstrip()]

                processed.extend([re.sub(r'\s+', ' ', x).lower() for x in p if x])# replace multiple blank

            return list(set(processed))

        return word.lower()


    # unchanged
    def compute_reward(self, text: str) -> float:
        """Compute lexical reward for a text."""
        tokens_no_stop, tokens_with_stop = self._preprocess_text(text)

        score = 0
        negative_score = 0
        matched = []
        negative_matched = []

        #TODO: update the matching algorithm
        # Match single words
        for token in tokens_no_stop:
            for w in self.word_list:
                # is word and exactly the same
                if isinstance(w, str) and token == w:
                    matched.append(token)
                    score += self.word_reward_dict[self.word_list.index(w)]
                    break

            for w in self.negative_word_list:
                if isinstance(w, str) and token == w:
                    negative_matched.append(token)
                    negative_score -= 1
                    break

        # Match phrases
        text_to_match = " ".join(tokens_with_stop)
        for idx, w in enumerate(self.word_list):
            if isinstance(w, list):
                if len(w) == 1:  # continuous phrase
                    if w[0] in text_to_match:
                        matched.append(w[0])
                        multiplier = 1.3 if len(w[0].split()) > 1 else 1.0
                        score += multiplier * self.word_reward_dict[idx]
                else:  # discontinuous phrase
                    matched_parts = 0
                    remaining_text = text_to_match
                    for part in w:
                        if len(part.split()) > 1 and part in remaining_text:
                            matched_parts += 1
                            remaining_text = "".join(remaining_text.split(part)[1:])

                    if matched_parts > 0:
                        score += 1.3 * (matched_parts / len(w)) * self.word_reward_dict[idx]

        final_score = score + negative_score
        return final_score / len(tokens_no_stop) if tokens_no_stop else 1.0


    # changed lemmatizer.lemmatize --> _lemmatize and stopwords
    def _preprocess_text(self, text: str) -> Tuple[List[str], List[str]]:
        """Preprocess text for matching."""
        stop_words = RU_STOP
        tokens = word_tokenize(text.lower())
        tokens = [w for w in tokens if w not in string.punctuation]

        tokens_no_stop = [w for w in tokens if w not in stop_words]
        tokens_no_stop = [self._lemmatize(w) for w in tokens_no_stop]

        tokens_with_stop = [self._lemmatize(w) for w in tokens]

        return tokens_no_stop, tokens_with_stop


    # unchanged
    def update_rewards(self, generations: List[str]):
        """Update reward weights based on word frequencies in generations."""
        # after each epoch, adjust the score for each matched token #
        # based on the tokens frequency, and IDF #
        # frequency is calculated globally, is the overall count for the token #
        # IDF is to encourage more generations to have rare words,
        # common words across all generations will have less score
        # ### but we no not want word appear only in one generation

        self.word_freq_dict = {idx: 0 for idx in range(len(self.word_list))}

        for text in tqdm(generations):
            tokens_no_stop, tokens_with_stop = self._preprocess_text(text)
            text_to_match = " ".join(tokens_with_stop)

            # Count word frequencies
            for idx, w in enumerate(self.word_list):
                if isinstance(w, str):
                    if w in tokens_no_stop:
                        self.word_freq_dict[idx] += 1
                else:  # phrase
                    if len(w) == 1:
                        if w[0] in text_to_match:
                            self.word_freq_dict[idx] += 1
                    else:
                        matched = False
                        for part in w:
                            if len(part.split()) > 1 and part in text_to_match:
                                matched = True
                        if matched:
                            self.word_freq_dict[idx] += 1

        # Update rewards based on frequencies
        total_words = sum(self.word_freq_dict.values())
        for idx in self.word_freq_dict:
            tf = self.word_freq_dict[idx] / total_words if total_words > 0 else 0
            if tf >= 1/len(self.word_list):
                self.word_reward_dict[idx] = 2**(-10*tf)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


### Config 

In [5]:
from dataclasses import dataclass
from typing import Optional, List, Literal


@dataclass
class PairwiseRewardModelConfig:
    model_name: str = "ai-forever/rugpt3small_based_on_gpt2"
    num_labels: int = 1
    level: str = "A" # "A", "B", or "C"
    train_batch_size: int = 32
    eval_batch_size: int = 32
    learning_rate: float = 1.41e-5
    # changed to 5
    num_train_epochs: int = 5
    gradient_accumulation_steps: int = 2
    output_dir: str = "./reward_model/"
    max_length: int = 128
    wandb_project: str = "CEFR-RewardModel"
    wandb_exp_name: Optional[str] = None


@dataclass
class RLTrainingConfig:
    training_sentences_path: Optional[str] = None
    model_name: str = "microsoft/Phi-3-mini-4k-instruct"
    level: str = "A" # "A", "B", or "C"
    batch_size: int = 48
    mini_batch_size: int = 16
    generate_bs: int = 32
    learning_rate: float = 3e-5
    ppo_epochs: int = 10
    max_new_tokens: int = 64
    gradient_accumulation_steps: int = 3
    output_dir: str = "./rl_model/"
    wandb_project: str = "PPO-training-sentence-simplification"
    wandb_exp_name: Optional[str] = None
    word_list_path: str = "./new_vocabulary.csv"
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.2

@dataclass
class BeamSearchConfig:
    base_model_path: str
    condition_model_path: str
    condition_tokenizer_path: str
    level: int  # CEFR level 1-6
    condition_type: Literal["bert", "gpt2"]
    condition_lambda: float = 0.8
    topk: int = 200
    num_beams: int = 5
    max_new_tokens: int = 100
    do_sample: bool = False

### Pairwise reward model 

In [6]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from typing import List, Dict, Optional
from tqdm import tqdm
from datasets import Dataset
from trl import RewardTrainer, RewardConfig
#from config import PairwiseRewardModelConfig
import os
#from data_utils import load_cefr_data

class CEFRRewardModel:
    def __init__(self, config: PairwiseRewardModelConfig):
        self.config = config
        self.model = self._init_model()
        self.tokenizer = self._init_tokenizer()


    def _init_model(self):
        model = AutoModelForSequenceClassification.from_pretrained(
            self.config.model_name,
            num_labels=self.config.num_labels,
            #torch_dtype=torch.bfloat16,
            #attn_implementation="flash_attention_2"
            #torch_dtype=torch.float16,
            attn_implementation="eager"
        )
        model.config.pad_token_id = model.config.eos_token_id
        return model.to('cuda')

    def _init_tokenizer(self):
        tokenizer = AutoTokenizer.from_pretrained(self.config.model_name)
        tokenizer.pad_token = tokenizer.eos_token
        return tokenizer

    def prepare_dataset(self, chosen_texts: List[str], rejected_texts: List[str]) -> Dataset:
        dataset = Dataset.from_dict({
            "chosen": chosen_texts,
            "rejected": rejected_texts
        })

        def preprocess(examples):
            new_examples = {
                "input_ids_chosen": [],
                "attention_mask_chosen": [],
                "input_ids_rejected": [],
                "attention_mask_rejected": [],
            }
            for chosen, rejected in zip(examples["chosen"], examples["rejected"]):
                tokenized_chosen = self.tokenizer(chosen)
                #tokenized_chosen = self.tokenizer(chosen, truncation=True, max_length=self.config.max_length)
                tokenized_rejected = self.tokenizer(rejected)
                #tokenized_rejected = self.tokenizer(rejected, truncation=True, max_length=self.config.max_length)

                new_examples["input_ids_chosen"].append(tokenized_chosen["input_ids"])
                new_examples["attention_mask_chosen"].append(tokenized_chosen["attention_mask"])
                new_examples["input_ids_rejected"].append(tokenized_rejected["input_ids"])
                new_examples["attention_mask_rejected"].append(tokenized_rejected["attention_mask"])

            return new_examples
        dataset = dataset.map(preprocess, batched=True, num_proc=torch.cuda.device_count())
        dataset = dataset.filter(lambda x: len(x["input_ids_chosen"]) <= self.config.max_length and len(x["input_ids_rejected"]) <= self.config.max_length)
        return dataset

    def train(self):
        print("Loading CEFR data... ")
        data = load_cefr_data(self.config.level, mode="reward")

        train_dataset = self.prepare_dataset(data["train"]["chosen"], data["train"]["rejected"])
        dev_dataset = self.prepare_dataset(data["dev"]["chosen"], data["dev"]["rejected"])
        eval_dataset = self.prepare_dataset(data["eval"]["chosen"], data["eval"]["rejected"])

        os.environ["WANDB_PROJECT"]=self.config.wandb_project
        os.environ['WANDB_WATCH'] = 'false'  # used in Trainer
        os.environ['WANDB_NAME'] = 'reward-model'  # used in Trainer
        training_args = RewardConfig(
            per_device_train_batch_size=self.config.train_batch_size,
            per_device_eval_batch_size=self.config.eval_batch_size,
            learning_rate=self.config.learning_rate,
            num_train_epochs=self.config.num_train_epochs,
            gradient_accumulation_steps=self.config.gradient_accumulation_steps,
            output_dir=self.config.output_dir,
            #bf16=True,
            #fp16=True, # new
            report_to = "wandb",
            logging_steps=1,
            eval_steps=10,
            max_length=self.config.max_length, # new
            remove_unused_columns=False, # new
        )

        trainer = RewardTrainer(
            model=self.model,
            #processing_class=self.tokenizer,
            tokenizer=self.tokenizer,
            train_dataset=train_dataset,
            eval_dataset=dev_dataset,
            args=training_args
        )

        trainer.train()
        trainer.evaluate(eval_dataset=eval_dataset)
        trainer.save_model(self.config.output_dir)

    def compute_rewards(self, texts: List[str]) -> torch.Tensor:
        """Compute rewards for generated texts."""
        rewards = []

        with torch.no_grad():
            for i in range(0, len(texts), self.config.eval_batch_size):
                batch = texts[i:i + self.config.eval_batch_size]
                inputs = self.tokenizer(batch, return_tensors="pt", padding=True)
                inputs = {k: v.to("cuda") for k, v in inputs.items()}

                logits = torch.sigmoid(self.model(**inputs).logits)
                rewards.append(logits)
        res = torch.vstack(rewards).cpu()
        res = res.to(torch.float32)

        return res

    @classmethod
    def load_model(cls, path: str):
        """Load a saved reward model."""
        model = AutoModelForSequenceClassification.from_pretrained(
            path,
            num_labels=1,
            #torch_dtype=torch.bfloat16,
            # attn_implementation="flash_attention_2"
            torch_dtype=torch.float16,
            attn_implementation="eager"
        )
        tokenizer = AutoTokenizer.from_pretrained(path)
        tokenizer.pad_token = tokenizer.eos_token

        model.config.pad_token_id = model.config.eos_token_id
        model.to('cuda')
        model.eval()

        return model, tokenizer

## Reward Evaluation

In [7]:
import numpy as np

def get_pairwise_scores(reward_model, level: str, limit: int = None):
    data = load_cefr_data(level, mode="reward")
    chosen = data["eval"]["chosen"]
    rejected = data["eval"]["rejected"]

    if limit is not None:
        chosen = chosen[:limit]
        rejected = rejected[:limit]

    chosen_scores = reward_model.compute_rewards(chosen).squeeze(-1).cpu().numpy()
    rejected_scores = reward_model.compute_rewards(rejected).squeeze(-1).cpu().numpy()
    return chosen_scores, rejected_scores


def pairwise_accuracy(chosen_scores, rejected_scores):
    diffs = chosen_scores - rejected_scores
    wins = np.sum(diffs > 0)
    ties = np.sum(diffs == 0)
    total = len(diffs)
    return {
        "pairwise_accuracy": float(wins / total),
        #"ties": int(ties),
        "n_pairs": int(total),
    }


def pairwise_margin(chosen_scores, rejected_scores):
    diffs = chosen_scores - rejected_scores
    return {
        "mean_margin": float(np.mean(diffs)),
        #"std_margin": float(np.std(diffs)),
    }


def bucket_mean_scores(reward_model, limit: int = None):
    A_texts = load_cefr_data("A", mode="rl")
    B_texts = load_cefr_data("B", mode="rl")
    C_texts = load_cefr_data("C", mode="rl")

    if limit is not None:
        A_texts = A_texts[:limit]
        B_texts = B_texts[:limit]
        C_texts = C_texts[:limit]

    return {
        "A_mean": float(reward_model.compute_rewards(A_texts).mean().item()),
        "B_mean": float(reward_model.compute_rewards(B_texts).mean().item()),
        "C_mean": float(reward_model.compute_rewards(C_texts).mean().item()),
    }


def evaluate_reward_model(reward_model, level: str, limit: int = None):
    chosen_scores, rejected_scores = get_pairwise_scores(reward_model, level, limit=limit)

    results = {"level": level}
    results.update(pairwise_accuracy(chosen_scores, rejected_scores))
    results.update(pairwise_margin(chosen_scores, rejected_scores))
    results.update(bucket_mean_scores(reward_model, limit=limit))

    return results

# Train

### Train_reward_model

## 2 epochs - A

In [ ]:
#from modeling.pairwise_reward_model import CEFRRewardModel
#from config import PairwiseRewardModelConfig
import torch
import argparse
import warnings

# changed for colab
def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--level", type=str, default="A")
    parser.add_argument("--model_name", type=str, default="ai-forever/rugpt3small_based_on_gpt2")
    parser.add_argument("--num_labels", type=int, default=1)
    parser.add_argument("--train_batch_size", type=int, default=32)
    parser.add_argument("--eval_batch_size", type=int, default=32)
    parser.add_argument("--learning_rate", type=float, default=1.41e-5)
    parser.add_argument("--num_train_epochs", type=int, default=2)
    parser.add_argument("--gradient_accumulation_steps", type=int, default=2)
    parser.add_argument("--output_dir", type=str, default="./reward_model/")
    parser.add_argument("--dataset_num_proc", type=int, default=64)
    parser.add_argument("--max_length", type=int, default=128)
    parser.add_argument("--wandb_project", type=str, default="CEFR-RewardModel")
    parser.add_argument("--wandb_exp_name", type=str, default=None)
    # changed for colab (was return parser.parse_args())
    return parser.parse_known_args()[0]


if __name__ == "__main__":

    if torch.cuda.device_count() > 1:
        warnings.warn(f"Multiple GPU Training has not been tested")
    args = parse_args()

    print("MODEL:", args.model_name)
    print("LEVEL:", args.level)
    print("EPOCHS:", args.num_train_epochs)

    pairwise_reward_model_config = PairwiseRewardModelConfig(
        model_name=args.model_name,
        num_labels=args.num_labels,
        level=args.level,
        train_batch_size=args.train_batch_size,
        eval_batch_size=args.eval_batch_size,
        learning_rate=args.learning_rate,
        num_train_epochs=args.num_train_epochs,
        gradient_accumulation_steps=args.gradient_accumulation_steps,
        output_dir=args.output_dir,
        max_length=args.max_length,
        wandb_project=args.wandb_project,
        wandb_exp_name=args.wandb_exp_name
    )

    pairwise_reward_model = CEFRRewardModel(pairwise_reward_model_config)
    pairwise_reward_model.train()

MODEL: ai-forever/rugpt3small_based_on_gpt2
LEVEL: A
EPOCHS: 2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/720 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/551M [00:00<?, ?B/s]

Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at ai-forever/rugpt3small_based_on_gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:2717: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss
1,0.727100
2,0.627900
3,0.597100
4,0.575300
5,0.499000
6,0.480100
7,0.609800
8,0.438300
9,0.425500
10,0.436500


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ chosen_text                                   ┃ rejected_text                                ┃ logits           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Думали ли читатели рассказов Рэя Бредбери,    │ Слово «экология» знают сейчас все люди. Как  │ [0.9082, 0.0918] │
│ что в нашей жизни будут «социальные сети» и   │ правило, его используют для характеристики   │                  │
│ мобильные телефоны, о которых он писал?       │ условий проживания: «хорошая (плохая)        │                  │
│                                               │ экология», «в нашем районе – ужасная         │                  │
│                                               │ экология». Эти слова понятны всем.           │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ В 17:45 я иду в супермаркет и покупаю         │ В 1970 году с благословения Американской     │ [0.993, 0.007]   │
│ продукты. Потом я иду домой и готовлю ужин.   │ академии религии Гэскин едет в тур по        │                  │
│ До ужина я ещё убираю квартиру. В 20:00 я     │ Америке — беседовать о духовности с          │                  │
│ ужинаю.                                       │ молодёжью. За ним, на машинах и школьных     │                  │
│                                               │ автобусах, едут его последователи.           │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ Я помню, несколько лет назад я была в банке.  │ Потратить деньги на какую-нибудь не очень    │ [0.9975, 0.0025] │
│ Там работал очень красивый мужчина. Мы        │ нужную вещь значит забыть на пару часов о    │                  │
│ познакомились, и он пригласил меня в кино.    │ проблемах в отношениях и на работе.          │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ Вы его, конечно, тоже знаете, но называете    │ Мы отдалились от природы, поэтому не ценим и │ [0.9383, 0.0617] │
│ «русский салат». Ингредиенты в «Оливье» очень │ не уважаем её. Природа отвечает нам тем же.  │                  │
│ простые. Это картофель, морковь, зелёный      │ Примером может служить изменение погоды и    │                  │
│ горошек, яйца, солёные огурцы и мясо. Иногда  │ климата на всей планете.                     │                  │
│ этот салат делают без мяса.                   │                                              │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ Этому городу уже 500 лет. В Плёсе есть        │ Г В этом сезоне нас чаще всего «лечили»      │ [0.8843, 0.1157] │
│ Дом-музей художника Исаака Ильича Левитана.   │ простуженные граждане, на остановке          │                  │
│ Этот известный художник очень любил природу,  │ ожидающие автобус. Помните, девушка          │                  │
│ поэтому он жил и работал в этом тихом зелёном │ «вылечила» от простуды стареющего рокера: —  │                  │
│ городе.                                       │ Апчхи. О, эта простуда.                      │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ На вертолёте он перелетел через Красное море, │ Гости могут принести вам хлеб и соль, это    │ [0.9668, 0.0332] │
│ прилетел в Аравию, а оттуда на машине поехал  │ символический жест по немецкой традиции.     │                  │
│ в Иран.                                       │ Подарки дарят разные.                        │                  │
├───────────────────────────────────────────────┼───────

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ chosen_text                                   ┃ rejected_text                                ┃ logits           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Думали ли читатели рассказов Рэя Бредбери,    │ Слово «экология» знают сейчас все люди. Как  │ [0.9082, 0.0918] │
│ что в нашей жизни будут «социальные сети» и   │ правило, его используют для характеристики   │                  │
│ мобильные телефоны, о которых он писал?       │ условий проживания: «хорошая (плохая)        │                  │
│                                               │ экология», «в нашем районе – ужасная         │                  │
│                                               │ экология». Эти слова понятны всем.           │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ В 17:45 я иду в супермаркет и покупаю         │ В 1970 году с благословения Американской     │ [0.993, 0.007]   │
│ продукты. Потом я иду домой и готовлю ужин.   │ академии религии Гэскин едет в тур по        │                  │
│ До ужина я ещё убираю квартиру. В 20:00 я     │ Америке — беседовать о духовности с          │                  │
│ ужинаю.                                       │ молодёжью. За ним, на машинах и школьных     │                  │
│                                               │ автобусах, едут его последователи.           │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ Я помню, несколько лет назад я была в банке.  │ Потратить деньги на какую-нибудь не очень    │ [0.9975, 0.0025] │
│ Там работал очень красивый мужчина. Мы        │ нужную вещь значит забыть на пару часов о    │                  │
│ познакомились, и он пригласил меня в кино.    │ проблемах в отношениях и на работе.          │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ Вы его, конечно, тоже знаете, но называете    │ Мы отдалились от природы, поэтому не ценим и │ [0.9383, 0.0617] │
│ «русский салат». Ингредиенты в «Оливье» очень │ не уважаем её. Природа отвечает нам тем же.  │                  │
│ простые. Это картофель, морковь, зелёный      │ Примером может служить изменение погоды и    │                  │
│ горошек, яйца, солёные огурцы и мясо. Иногда  │ климата на всей планете.                     │                  │
│ этот салат делают без мяса.                   │                                              │                  │
└───────────────────────────────────────────────┴──────────────────────────────────────────────┴──────────────────┘

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:2717: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


In [ ]:
texts = [
    "Я люблю читать книги.",
    "Сегодня хорошая погода.",
    "Мне нравится гулять в парке.",
    "Несмотря на то, что рассматриваемая проблематика обладает многослойной структурой, её интерпретация требует дополнительной методологической рефлексии.",
    "С учётом вышеизложенного представляется необходимым пересмотреть методологические основания исследования.",
    "Дискуссия о социокультурной обусловленности языковых практик требует междисциплинарного подхода."
]

scores = pairwise_reward_model.compute_rewards(texts)
for t, s in zip(texts, scores):
    print(float(s), t)

0.9983305335044861 Я люблю читать книги.
0.9804269075393677 Сегодня хорошая погода.
0.9906908273696899 Мне нравится гулять в парке.
0.09505032747983932 Несмотря на то, что рассматриваемая проблематика обладает многослойной структурой, её интерпретация требует дополнительной методологической рефлексии.
0.004655271302908659 С учётом вышеизложенного представляется необходимым пересмотреть методологические основания исследования.
0.03013445995748043 Дискуссия о социокультурной обусловленности языковых практик требует междисциплинарного подхода.


In [ ]:
results = evaluate_reward_model(pairwise_reward_model, level="A")
print(results)

{'level': 'A', 'pairwise_accuracy': 0.8835978835978836, 'n_pairs': 189, 'mean_margin': 0.4401126801967621, 'A_mean': 0.8947047591209412, 'B_mean': 0.48260441422462463, 'C_mean': 0.283855140209198}


In [ ]:
results["model"] = "rugpt"
results["epochs"] = 2
df_results = pd.DataFrame([results])
df_results.to_csv("reward_model_results_rugpt_2_A.csv", index=False)
df_results

,level,pairwise_accuracy,n_pairs,mean_margin,A_mean,B_mean,C_mean,model,epochs
0,A,0.883598,189,0.440113,0.894705,0.482604,0.283855,gpt2,2


## 2 epochs - B

In [8]:
#from modeling.pairwise_reward_model import CEFRRewardModel
#from config import PairwiseRewardModelConfig
import torch
import argparse
import warnings

# changed for colab
def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--level", type=str, default="B")
    parser.add_argument("--model_name", type=str, default="ai-forever/rugpt3small_based_on_gpt2")
    parser.add_argument("--num_labels", type=int, default=1)
    parser.add_argument("--train_batch_size", type=int, default=32)
    parser.add_argument("--eval_batch_size", type=int, default=32)
    parser.add_argument("--learning_rate", type=float, default=1.41e-5)
    parser.add_argument("--num_train_epochs", type=int, default=2)
    parser.add_argument("--gradient_accumulation_steps", type=int, default=2)
    parser.add_argument("--output_dir", type=str, default="./reward_model/")
    parser.add_argument("--dataset_num_proc", type=int, default=64)
    parser.add_argument("--max_length", type=int, default=128)
    parser.add_argument("--wandb_project", type=str, default="CEFR-RewardModel")
    parser.add_argument("--wandb_exp_name", type=str, default=None)
    # changed for colab (was return parser.parse_args())
    return parser.parse_known_args()[0]


if __name__ == "__main__":

    if torch.cuda.device_count() > 1:
        warnings.warn(f"Multiple GPU Training has not been tested")
    args = parse_args()

    print("MODEL:", args.model_name)
    print("LEVEL:", args.level)
    print("EPOCHS:", args.num_train_epochs)

    pairwise_reward_model_config = PairwiseRewardModelConfig(
        model_name=args.model_name,
        num_labels=args.num_labels,
        level=args.level,
        train_batch_size=args.train_batch_size,
        eval_batch_size=args.eval_batch_size,
        learning_rate=args.learning_rate,
        num_train_epochs=args.num_train_epochs,
        gradient_accumulation_steps=args.gradient_accumulation_steps,
        output_dir=args.output_dir,
        max_length=args.max_length,
        wandb_project=args.wandb_project,
        wandb_exp_name=args.wandb_exp_name
    )

    pairwise_reward_model = CEFRRewardModel(pairwise_reward_model_config)
    pairwise_reward_model.train()

MODEL: ai-forever/rugpt3small_based_on_gpt2
LEVEL: B
EPOCHS: 2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/720 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/551M [00:00<?, ?B/s]

Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at ai-forever/rugpt3small_based_on_gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:2717: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss
1,0.831400
2,0.736400
3,0.668500
4,0.721600
5,0.654400
6,0.709900
7,0.669800
8,0.661200
9,0.661500
10,0.633300


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ chosen_text                                   ┃ rejected_text                                ┃ logits           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Это значит, мы приезжаем к университету, вы   │ У меня есть подруга, я её очень люблю, и всё │ [0.4637, 0.5363] │
│ выходите из машины и идёте учиться. Отвозить  │ мне в ней нравится. У неё длинные густые     │                  │
│ или отвезти кого-то куда-то – это значит      │ волнистые каштановые волосы, высокий умный   │                  │
│ привезти человека в какое-то место и оставить │ лоб и вздёрнутый кверху задорный носик.      │                  │
│ там. Понятно?                                 │                                              │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ Форт № 11 «Дёнхофф» был построен в XIX веке и │ Сегодня это не просто миллионы селфи         │ [0.6943, 0.3057] │
│ хорошо сохранился. Здесь проходят экскурсии и │ туристов и моделей. Компании часто           │                  │
│ даже квесты. По территории форта спокойно     │ рекламируют и продают продукцию в            │                  │
│ гуляют кролики, которые дают посетителям себя │ «Инстаграме». Кстати, 68 процентов           │                  │
│ погладить.                                    │ пользователей в «Инстаграме» — женщины.      │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ Его пригласили на интервью в эту компанию и   │ Эта поездка помогла мне лучше узнать Россию  │ [0.8367, 0.1633] │
│ взяли на работу, потому что Сергей хорошо     │ и Украину, их историю и современность. Я     │                  │
│ знал английский язык.                         │ думаю, что я поеду в Крым ещё раз.           │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ Фотография с цветущим деревом во дворе жилого │ Она сказала, что Ульяна — очень серьёзная    │ [0.7974, 0.2026] │
│ дома размещена в социальных сетях в группе «Я │ девушка и у неё уже есть молодой человек.    │                  │
│ живу в Красноярске».                          │ Его зовут Владимир.                          │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ Однако, чтобы не обидеть уже немолодого       │ Спартаковцы ехали на футбол, и их болельщики │ [0.6949, 0.3051] │
│ абитуриента, девушки сказали ему, что у него  │ ехали на футбол. В метро было много людей. К │                  │
│ очень красивое и выразительное лицо, и        │ счастью, футболисты никого не потеряли. Все  │                  │
│ посоветовали ему поступать на актёрский       │ приехали на стадион вовремя.                 │                  │
│ факультет.                                    │                                              │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ Если у вашего собеседника есть домашний       │ И вы знаете, что случилось в этот вечер?!    │ [0.9713, 0.0287] │
│ любимец, он с удовольствием о нём расскажет   │ Анна тоже была на концерте. Это был также и  │                  │
│ и, может быть, покажет фотографии. хобби =    │ её любимый музыкант. Она не знала, что я     │                  │
│ увлечение. Всегда интересно говорить о хобби. │ хотел идти на концерт.                       │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ По мнению учёных, причиной экологической      │ Говоря

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ chosen_text                                   ┃ rejected_text                                ┃ logits           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Это значит, мы приезжаем к университету, вы   │ У меня есть подруга, я её очень люблю, и всё │ [0.4637, 0.5363] │
│ выходите из машины и идёте учиться. Отвозить  │ мне в ней нравится. У неё длинные густые     │                  │
│ или отвезти кого-то куда-то – это значит      │ волнистые каштановые волосы, высокий умный   │                  │
│ привезти человека в какое-то место и оставить │ лоб и вздёрнутый кверху задорный носик.      │                  │
│ там. Понятно?                                 │                                              │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ Форт № 11 «Дёнхофф» был построен в XIX веке и │ Сегодня это не просто миллионы селфи         │ [0.6943, 0.3057] │
│ хорошо сохранился. Здесь проходят экскурсии и │ туристов и моделей. Компании часто           │                  │
│ даже квесты. По территории форта спокойно     │ рекламируют и продают продукцию в            │                  │
│ гуляют кролики, которые дают посетителям себя │ «Инстаграме». Кстати, 68 процентов           │                  │
│ погладить.                                    │ пользователей в «Инстаграме» — женщины.      │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ Его пригласили на интервью в эту компанию и   │ Эта поездка помогла мне лучше узнать Россию  │ [0.8367, 0.1633] │
│ взяли на работу, потому что Сергей хорошо     │ и Украину, их историю и современность. Я     │                  │
│ знал английский язык.                         │ думаю, что я поеду в Крым ещё раз.           │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ Фотография с цветущим деревом во дворе жилого │ Она сказала, что Ульяна — очень серьёзная    │ [0.7974, 0.2026] │
│ дома размещена в социальных сетях в группе «Я │ девушка и у неё уже есть молодой человек.    │                  │
│ живу в Красноярске».                          │ Его зовут Владимир.                          │                  │
└───────────────────────────────────────────────┴──────────────────────────────────────────────┴──────────────────┘

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:2717: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


In [9]:
texts = [
    "Я люблю читать книги.",
    "Сегодня хорошая погода.",
    "Мне нравится гулять в парке.",
    "Несмотря на то, что рассматриваемая проблематика обладает многослойной структурой, её интерпретация требует дополнительной методологической рефлексии.",
    "С учётом вышеизложенного представляется необходимым пересмотреть методологические основания исследования.",
    "Дискуссия о социокультурной обусловленности языковых практик требует междисциплинарного подхода."
]

scores = pairwise_reward_model.compute_rewards(texts)
for t, s in zip(texts, scores):
    print(float(s), t)

0.039037931710481644 Я люблю читать книги.
0.16050927340984344 Сегодня хорошая погода.
0.3049352169036865 Мне нравится гулять в парке.
0.4095620810985565 Несмотря на то, что рассматриваемая проблематика обладает многослойной структурой, её интерпретация требует дополнительной методологической рефлексии.
0.46564415097236633 С учётом вышеизложенного представляется необходимым пересмотреть методологические основания исследования.
0.3463592529296875 Дискуссия о социокультурной обусловленности языковых практик требует междисциплинарного подхода.


In [10]:
results = evaluate_reward_model(pairwise_reward_model, level="B")
print(results)

{'level': 'B', 'pairwise_accuracy': 0.751219512195122, 'n_pairs': 410, 'mean_margin': 0.21784210205078125, 'A_mean': 0.24628077447414398, 'B_mean': 0.5709921717643738, 'C_mean': 0.47555774450302124}


In [11]:
results["model"] = "rugpt"
results["epochs"] = 2
df_results = pd.DataFrame([results])
df_results.to_csv("reward_model_results_rugpt_2_B.csv", index=False)
df_results

,level,pairwise_accuracy,n_pairs,mean_margin,A_mean,B_mean,C_mean,model,epochs
0,B,0.75122,410,0.217842,0.246281,0.570992,0.475558,rugpt,2


## 2 epochs - C

In [9]:
#from modeling.pairwise_reward_model import CEFRRewardModel
#from config import PairwiseRewardModelConfig
import torch
import argparse
import warnings

# changed for colab
def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--level", type=str, default="C")
    parser.add_argument("--model_name", type=str, default="ai-forever/rugpt3small_based_on_gpt2")
    parser.add_argument("--num_labels", type=int, default=1)
    parser.add_argument("--train_batch_size", type=int, default=32)
    parser.add_argument("--eval_batch_size", type=int, default=32)
    parser.add_argument("--learning_rate", type=float, default=1.41e-5)
    parser.add_argument("--num_train_epochs", type=int, default=2)
    parser.add_argument("--gradient_accumulation_steps", type=int, default=2)
    parser.add_argument("--output_dir", type=str, default="./reward_model/")
    parser.add_argument("--dataset_num_proc", type=int, default=64)
    parser.add_argument("--max_length", type=int, default=128)
    parser.add_argument("--wandb_project", type=str, default="CEFR-RewardModel")
    parser.add_argument("--wandb_exp_name", type=str, default=None)
    # changed for colab (was return parser.parse_args())
    return parser.parse_known_args()[0]


if __name__ == "__main__":

    if torch.cuda.device_count() > 1:
        warnings.warn(f"Multiple GPU Training has not been tested")
    args = parse_args()

    print("MODEL:", args.model_name)
    print("LEVEL:", args.level)
    print("EPOCHS:", args.num_train_epochs)

    pairwise_reward_model_config = PairwiseRewardModelConfig(
        model_name=args.model_name,
        num_labels=args.num_labels,
        level=args.level,
        train_batch_size=args.train_batch_size,
        eval_batch_size=args.eval_batch_size,
        learning_rate=args.learning_rate,
        num_train_epochs=args.num_train_epochs,
        gradient_accumulation_steps=args.gradient_accumulation_steps,
        output_dir=args.output_dir,
        max_length=args.max_length,
        wandb_project=args.wandb_project,
        wandb_exp_name=args.wandb_exp_name
    )

    pairwise_reward_model = CEFRRewardModel(pairwise_reward_model_config)
    pairwise_reward_model.train()

MODEL: ai-forever/rugpt3small_based_on_gpt2
LEVEL: C
EPOCHS: 2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at ai-forever/rugpt3small_based_on_gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loading CEFR data... 


Map:   0%|          | 0/945 [00:00<?, ? examples/s]

Filter:   0%|          | 0/945 [00:00<?, ? examples/s]

...

You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:2717: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss
1,0.772800
2,0.667500
3,0.596800
4,0.623800
5,0.656000
6,0.583600
7,0.588000
8,0.535400
9,0.517800
10,0.533300


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ chosen_text                                   ┃ rejected_text                                ┃ logits           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ МеМеМе) — первым поколением, сформировавшимся │ Сигареты, алкоголь, наркотики, соль, сахар,  │ [0.3592, 0.6408] │
│ при Интернете. ЯЯЯ взаимодействуют с миром    │ чипсы, газировка, энергетики (список можно   │                  │
│ круглые сутки, но в основном через экран,     │ продолжить) — всё это факторы, разрушающие   │                  │
│ фиксируя каждый свой шаг.                     │ здоровье. Отказаться от них трудно, но       │                  │
│                                               │ можно. Многое зависит от нашей воли и        │                  │
│                                               │ мотивации.                                   │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ Крупные банки и их филиалы предоставляют      │ Вот что рассказала о своём сыне мать         │ [0.94, 0.06]     │
│ отличные условия для работы молодых           │ чемпиона Клара Каспарова: — Гарик родился 13 │                  │
│ специалистов, предлагая высокую оплату их     │ апреля 1963 года. Рос он здоровым спокойным  │                  │
│ труда и возможность хорошо отдохнуть во время │ мальчиком.                                   │                  │
│ отпуска.                                      │                                              │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ Православными себя называют 63 % наших        │ У нас в деревне люди везде ездят на машинах: │ [0.9703, 0.0297] │
│ сограждан, мусульманами — 6 %, католиками и   │ в магазин, к врачу, на пляж, на работу. Это  │                  │
│ буддистами — по 1 %, последователей других    │ дорого, потому что машина дорогая и бензин   │                  │
│ традиционных конфессий и того меньше, а 16 %  │ дорогой.                                     │                  │
│ населения страны записались в атеисты.        │                                              │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ Затем разбивать их на куски и продавать       │ В 17:45 я иду в супермаркет и покупаю        │ [0.9918, 0.0082] │
│ крестьянам. Так часть крестьян уйдёт из       │ продукты. Потом я иду домой и готовлю ужин.  │                  │
│ общин, станет активнее переселяться в Сибирь. │ До ужина я ещё убираю квартиру. В 20:00 я    │                  │
│ Некоторые крепкие хозяева станут строить      │ ужинаю.                                      │                  │
│ хутора.                                       │                                              │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ Очевидно, если литературный язык определяется │ Слово «экология» знают сейчас все люди. Как  │ [0.7969, 0.2031] │
│ как общий язык письменности того или иного    │ правило, его используют для характеристики   │                  │
│ народа (В. В. Виноградов), то остальные его   │ условий проживания: «хорошая (плохая)        │                  │
│ разновидности носят более узкий (с точки      │ экология», «в нашем районе – ужасная         │                  │
│ зрения употребления) характер.                │ экология». Эти слова понятны всем.           │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ Знакомые размеры фотокарточек сделали         │ Люди п

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ chosen_text                                   ┃ rejected_text                                ┃ logits           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ МеМеМе) — первым поколением, сформировавшимся │ Сигареты, алкоголь, наркотики, соль, сахар,  │ [0.3592, 0.6408] │
│ при Интернете. ЯЯЯ взаимодействуют с миром    │ чипсы, газировка, энергетики (список можно   │                  │
│ круглые сутки, но в основном через экран,     │ продолжить) — всё это факторы, разрушающие   │                  │
│ фиксируя каждый свой шаг.                     │ здоровье. Отказаться от них трудно, но       │                  │
│                                               │ можно. Многое зависит от нашей воли и        │                  │
│                                               │ мотивации.                                   │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ Крупные банки и их филиалы предоставляют      │ Вот что рассказала о своём сыне мать         │ [0.94, 0.06]     │
│ отличные условия для работы молодых           │ чемпиона Клара Каспарова: — Гарик родился 13 │                  │
│ специалистов, предлагая высокую оплату их     │ апреля 1963 года. Рос он здоровым спокойным  │                  │
│ труда и возможность хорошо отдохнуть во время │ мальчиком.                                   │                  │
│ отпуска.                                      │                                              │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ Православными себя называют 63 % наших        │ У нас в деревне люди везде ездят на машинах: │ [0.9703, 0.0297] │
│ сограждан, мусульманами — 6 %, католиками и   │ в магазин, к врачу, на пляж, на работу. Это  │                  │
│ буддистами — по 1 %, последователей других    │ дорого, потому что машина дорогая и бензин   │                  │
│ традиционных конфессий и того меньше, а 16 %  │ дорогой.                                     │                  │
│ населения страны записались в атеисты.        │                                              │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ Затем разбивать их на куски и продавать       │ В 17:45 я иду в супермаркет и покупаю        │ [0.9918, 0.0082] │
│ крестьянам. Так часть крестьян уйдёт из       │ продукты. Потом я иду домой и готовлю ужин.  │                  │
│ общин, станет активнее переселяться в Сибирь. │ До ужина я ещё убираю квартиру. В 20:00 я    │                  │
│ Некоторые крепкие хозяева станут строить      │ ужинаю.                                      │                  │
│ хутора.                                       │                                              │                  │
└───────────────────────────────────────────────┴──────────────────────────────────────────────┴──────────────────┘

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:2717: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


In [10]:
texts = [
    "Я люблю читать книги.",
    "Сегодня хорошая погода.",
    "Мне нравится гулять в парке.",
    "Несмотря на то, что рассматриваемая проблематика обладает многослойной структурой, её интерпретация требует дополнительной методологической рефлексии.",
    "С учётом вышеизложенного представляется необходимым пересмотреть методологические основания исследования.",
    "Дискуссия о социокультурной обусловленности языковых практик требует междисциплинарного подхода."
]

scores = pairwise_reward_model.compute_rewards(texts)
for t, s in zip(texts, scores):
    print(float(s), t)

0.033042773604393005 Я люблю читать книги.
0.053913358598947525 Сегодня хорошая погода.
0.03546735271811485 Мне нравится гулять в парке.
0.7766793370246887 Несмотря на то, что рассматриваемая проблематика обладает многослойной структурой, её интерпретация требует дополнительной методологической рефлексии.
0.8545243740081787 С учётом вышеизложенного представляется необходимым пересмотреть методологические основания исследования.
0.8346555829048157 Дискуссия о социокультурной обусловленности языковых практик требует междисциплинарного подхода.


In [11]:
results = evaluate_reward_model(pairwise_reward_model, level="C")
print(results)

{'target_level': 'C', 'pairwise_accuracy': 0.865546218487395, 'tie_rate': 0.0, 'loss_rate': 0.13445378151260504, 'n_pairs': 119, 'mean_margin': 0.3058505654335022, 'std_margin': 0.28124526143074036, 'median_margin': 0.2985902428627014, 'A_mean': 0.09946602582931519, 'B_mean': 0.2591073513031006, 'C_mean': 0.4985193908214569, 'A_std': 0.1040613204240799, 'B_std': 0.18758131563663483, 'C_std': 0.22719290852546692, 'monotonic_ABC': True, 'spearman_cefr': 0.6004140415558545, 'spearman_pvalue': 0.0}


In [12]:
results["model"] = "rugpt"
results["epochs"] = 2
df_results = pd.DataFrame([results])
df_results.to_csv("reward_model_results_rugpt_2_C.csv", index=False)
df_results

,target_level,pairwise_accuracy,tie_rate,loss_rate,n_pairs,mean_margin,std_margin,median_margin,A_mean,B_mean,C_mean,A_std,B_std,C_std,monotonic_ABC,spearman_cefr,spearman_pvalue,model,epochs
0,C,0.865546,0.0,0.134454,119,0.305851,0.281245,0.29859,0.099466,0.259107,0.498519,0.104061,0.187581,0.227193,True,0.600414,0.0,rugpt,2


## 5 epochs - A

In [ ]:
#from modeling.pairwise_reward_model import CEFRRewardModel
#from config import PairwiseRewardModelConfig
import torch
import argparse
import warnings

# changed for colab
def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--level", type=str, default="A")
    parser.add_argument("--model_name", type=str, default="ai-forever/rugpt3small_based_on_gpt2")
    parser.add_argument("--num_labels", type=int, default=1)
    parser.add_argument("--train_batch_size", type=int, default=32)
    parser.add_argument("--eval_batch_size", type=int, default=32)
    parser.add_argument("--learning_rate", type=float, default=1.41e-5)
    parser.add_argument("--num_train_epochs", type=int, default=5)
    parser.add_argument("--gradient_accumulation_steps", type=int, default=2)
    parser.add_argument("--output_dir", type=str, default="./reward_model/")
    parser.add_argument("--dataset_num_proc", type=int, default=64)
    parser.add_argument("--max_length", type=int, default=128)
    parser.add_argument("--wandb_project", type=str, default="CEFR-RewardModel")
    parser.add_argument("--wandb_exp_name", type=str, default=None)
    # changed for colab (was return parser.parse_args())
    return parser.parse_known_args()[0]


if __name__ == "__main__":

    if torch.cuda.device_count() > 1:
        warnings.warn(f"Multiple GPU Training has not been tested")
    args = parse_args()
    pairwise_reward_model_config = PairwiseRewardModelConfig(
        model_name=args.model_name,
        num_labels=args.num_labels,
        level=args.level,
        train_batch_size=args.train_batch_size,
        eval_batch_size=args.eval_batch_size,
        learning_rate=args.learning_rate,
        num_train_epochs=args.num_train_epochs,
        gradient_accumulation_steps=args.gradient_accumulation_steps,
        output_dir=args.output_dir,
        max_length=args.max_length,
        wandb_project=args.wandb_project,
        wandb_exp_name=args.wandb_exp_name
    )

    pairwise_reward_model = CEFRRewardModel(pairwise_reward_model_config)
    pairwise_reward_model.train()

Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at ai-forever/rugpt3small_based_on_gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loading CEFR data... 


Map:   0%|          | 0/1519 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1519 [00:00<?, ? examples/s]

Map:   0%|          | 0/191 [00:00<?, ? examples/s]

...

You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:2717: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss
1,0.660900
2,0.652200
3,0.535500
4,0.540200
5,0.495900
6,0.417400
7,0.507700
8,0.408500
9,0.365700
10,0.455700


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ chosen_text                                   ┃ rejected_text                                ┃ logits           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Думали ли читатели рассказов Рэя Бредбери,    │ Слово «экология» знают сейчас все люди. Как  │ [0.9926, 0.0074] │
│ что в нашей жизни будут «социальные сети» и   │ правило, его используют для характеристики   │                  │
│ мобильные телефоны, о которых он писал?       │ условий проживания: «хорошая (плохая)        │                  │
│                                               │ экология», «в нашем районе – ужасная         │                  │
│                                               │ экология». Эти слова понятны всем.           │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ В 17:45 я иду в супермаркет и покупаю         │ В 1970 году с благословения Американской     │ [0.9995, 0.0005] │
│ продукты. Потом я иду домой и готовлю ужин.   │ академии религии Гэскин едет в тур по        │                  │
│ До ужина я ещё убираю квартиру. В 20:00 я     │ Америке — беседовать о духовности с          │                  │
│ ужинаю.                                       │ молодёжью. За ним, на машинах и школьных     │                  │
│                                               │ автобусах, едут его последователи.           │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ Я помню, несколько лет назад я была в банке.  │ Потратить деньги на какую-нибудь не очень    │ [0.9933, 0.0067] │
│ Там работал очень красивый мужчина. Мы        │ нужную вещь значит забыть на пару часов о    │                  │
│ познакомились, и он пригласил меня в кино.    │ проблемах в отношениях и на работе.          │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ Вы его, конечно, тоже знаете, но называете    │ Мы отдалились от природы, поэтому не ценим и │ [0.9957, 0.0043] │
│ «русский салат». Ингредиенты в «Оливье» очень │ не уважаем её. Природа отвечает нам тем же.  │                  │
│ простые. Это картофель, морковь, зелёный      │ Примером может служить изменение погоды и    │                  │
│ горошек, яйца, солёные огурцы и мясо. Иногда  │ климата на всей планете.                     │                  │
│ этот салат делают без мяса.                   │                                              │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ Этому городу уже 500 лет. В Плёсе есть        │ Г В этом сезоне нас чаще всего «лечили»      │ [0.9788, 0.0212] │
│ Дом-музей художника Исаака Ильича Левитана.   │ простуженные граждане, на остановке          │                  │
│ Этот известный художник очень любил природу,  │ ожидающие автобус. Помните, девушка          │                  │
│ поэтому он жил и работал в этом тихом зелёном │ «вылечила» от простуды стареющего рокера: —  │                  │
│ городе.                                       │ Апчхи. О, эта простуда.                      │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ На вертолёте он перелетел через Красное море, │ Гости могут принести вам хлеб и соль, это    │ [0.9939, 0.0061] │
│ прилетел в Аравию, а оттуда на машине поехал  │ символический жест по немецкой традиции.     │                  │
│ в Иран.                                       │ Подарки дарят разные.                        │                  │
├───────────────────────────────────────────────┼───────

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ chosen_text                                   ┃ rejected_text                                ┃ logits           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Думали ли читатели рассказов Рэя Бредбери,    │ Слово «экология» знают сейчас все люди. Как  │ [0.9926, 0.0074] │
│ что в нашей жизни будут «социальные сети» и   │ правило, его используют для характеристики   │                  │
│ мобильные телефоны, о которых он писал?       │ условий проживания: «хорошая (плохая)        │                  │
│                                               │ экология», «в нашем районе – ужасная         │                  │
│                                               │ экология». Эти слова понятны всем.           │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ В 17:45 я иду в супермаркет и покупаю         │ В 1970 году с благословения Американской     │ [0.9995, 0.0005] │
│ продукты. Потом я иду домой и готовлю ужин.   │ академии религии Гэскин едет в тур по        │                  │
│ До ужина я ещё убираю квартиру. В 20:00 я     │ Америке — беседовать о духовности с          │                  │
│ ужинаю.                                       │ молодёжью. За ним, на машинах и школьных     │                  │
│                                               │ автобусах, едут его последователи.           │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ Я помню, несколько лет назад я была в банке.  │ Потратить деньги на какую-нибудь не очень    │ [0.9933, 0.0067] │
│ Там работал очень красивый мужчина. Мы        │ нужную вещь значит забыть на пару часов о    │                  │
│ познакомились, и он пригласил меня в кино.    │ проблемах в отношениях и на работе.          │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ Вы его, конечно, тоже знаете, но называете    │ Мы отдалились от природы, поэтому не ценим и │ [0.9957, 0.0043] │
│ «русский салат». Ингредиенты в «Оливье» очень │ не уважаем её. Природа отвечает нам тем же.  │                  │
│ простые. Это картофель, морковь, зелёный      │ Примером может служить изменение погоды и    │                  │
│ горошек, яйца, солёные огурцы и мясо. Иногда  │ климата на всей планете.                     │                  │
│ этот салат делают без мяса.                   │                                              │                  │
└───────────────────────────────────────────────┴──────────────────────────────────────────────┴──────────────────┘

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:2717: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


In [ ]:
texts = [
    "Я люблю читать книги.",
    "Сегодня хорошая погода.",
    "Мне нравится гулять в парке.",
    "Несмотря на то, что рассматриваемая проблематика обладает многослойной структурой, её интерпретация требует дополнительной методологической рефлексии.",
    "С учётом вышеизложенного представляется необходимым пересмотреть методологические основания исследования.",
    "Дискуссия о социокультурной обусловленности языковых практик требует междисциплинарного подхода."
]

scores = pairwise_reward_model.compute_rewards(texts)
for t, s in zip(texts, scores):
    print(float(s), t)

0.9999548196792603 Я люблю читать книги.
0.9997864365577698 Сегодня хорошая погода.
0.998775064945221 Мне нравится гулять в парке.
0.002800342161208391 Несмотря на то, что рассматриваемая проблематика обладает многослойной структурой, её интерпретация требует дополнительной методологической рефлексии.
0.00037846105988137424 С учётом вышеизложенного представляется необходимым пересмотреть методологические основания исследования.
0.0006221923395060003 Дискуссия о социокультурной обусловленности языковых практик требует междисциплинарного подхода.


In [ ]:
results = evaluate_reward_model(pairwise_reward_model, level="A")
print(results)

{'level': 'A', 'pairwise_accuracy': 0.9206349206349206, 'n_pairs': 189, 'mean_margin': 0.5501500964164734, 'A_mean': 0.9646997451782227, 'B_mean': 0.46428996324539185, 'C_mean': 0.22893282771110535}


In [ ]:
results["model"] = "rugpt"
results["epochs"] = 5
df_results = pd.DataFrame([results])
df_results.to_csv("reward_model_results_rugpt_5_A.csv", index=False)
df_results

,level,pairwise_accuracy,n_pairs,mean_margin,A_mean,B_mean,C_mean,model,epochs
0,A,0.920635,189,0.55015,0.9647,0.46429,0.228933,rugpt,5


## 5 epochs - B

In [8]:
#from modeling.pairwise_reward_model import CEFRRewardModel
#from config import PairwiseRewardModelConfig
import torch
import argparse
import warnings

# changed for colab
def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--level", type=str, default="B")
    parser.add_argument("--model_name", type=str, default="ai-forever/rugpt3small_based_on_gpt2")
    parser.add_argument("--num_labels", type=int, default=1)
    parser.add_argument("--train_batch_size", type=int, default=32)
    parser.add_argument("--eval_batch_size", type=int, default=32)
    parser.add_argument("--learning_rate", type=float, default=1.41e-5)
    parser.add_argument("--num_train_epochs", type=int, default=5)
    parser.add_argument("--gradient_accumulation_steps", type=int, default=2)
    parser.add_argument("--output_dir", type=str, default="./reward_model/")
    parser.add_argument("--dataset_num_proc", type=int, default=64)
    parser.add_argument("--max_length", type=int, default=128)
    parser.add_argument("--wandb_project", type=str, default="CEFR-RewardModel")
    parser.add_argument("--wandb_exp_name", type=str, default=None)
    # changed for colab (was return parser.parse_args())
    return parser.parse_known_args()[0]


if __name__ == "__main__":

    if torch.cuda.device_count() > 1:
        warnings.warn(f"Multiple GPU Training has not been tested")
    args = parse_args()
    print("MODEL:", args.model_name)
    print("LEVEL:", args.level)
    print("EPOCHS:", args.num_train_epochs)
    pairwise_reward_model_config = PairwiseRewardModelConfig(
        model_name=args.model_name,
        num_labels=args.num_labels,
        level=args.level,
        train_batch_size=args.train_batch_size,
        eval_batch_size=args.eval_batch_size,
        learning_rate=args.learning_rate,
        num_train_epochs=args.num_train_epochs,
        gradient_accumulation_steps=args.gradient_accumulation_steps,
        output_dir=args.output_dir,
        max_length=args.max_length,
        wandb_project=args.wandb_project,
        wandb_exp_name=args.wandb_exp_name
    )

    pairwise_reward_model = CEFRRewardModel(pairwise_reward_model_config)
    pairwise_reward_model.train()

MODEL: ai-forever/rugpt3small_based_on_gpt2
LEVEL: B
EPOCHS: 5


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at ai-forever/rugpt3small_based_on_gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loading CEFR data... 


Map:   0%|          | 0/3273 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3273 [00:00<?, ? examples/s]

...

You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:2717: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss
1,0.699500
2,0.697100
3,0.660400
4,0.680200
5,0.662000
6,0.629800
7,0.696100
8,0.585700
9,0.667700
10,0.557800


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ chosen_text                                   ┃ rejected_text                                ┃ logits           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Это значит, мы приезжаем к университету, вы   │ У меня есть подруга, я её очень люблю, и всё │ [0.8465, 0.1535] │
│ выходите из машины и идёте учиться. Отвозить  │ мне в ней нравится. У неё длинные густые     │                  │
│ или отвезти кого-то куда-то – это значит      │ волнистые каштановые волосы, высокий умный   │                  │
│ привезти человека в какое-то место и оставить │ лоб и вздёрнутый кверху задорный носик.      │                  │
│ там. Понятно?                                 │                                              │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ Форт № 11 «Дёнхофф» был построен в XIX веке и │ Сегодня это не просто миллионы селфи         │ [0.7655, 0.2345] │
│ хорошо сохранился. Здесь проходят экскурсии и │ туристов и моделей. Компании часто           │                  │
│ даже квесты. По территории форта спокойно     │ рекламируют и продают продукцию в            │                  │
│ гуляют кролики, которые дают посетителям себя │ «Инстаграме». Кстати, 68 процентов           │                  │
│ погладить.                                    │ пользователей в «Инстаграме» — женщины.      │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ Его пригласили на интервью в эту компанию и   │ Эта поездка помогла мне лучше узнать Россию  │ [0.168, 0.832]   │
│ взяли на работу, потому что Сергей хорошо     │ и Украину, их историю и современность. Я     │                  │
│ знал английский язык.                         │ думаю, что я поеду в Крым ещё раз.           │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ Фотография с цветущим деревом во дворе жилого │ Она сказала, что Ульяна — очень серьёзная    │ [0.8519, 0.1481] │
│ дома размещена в социальных сетях в группе «Я │ девушка и у неё уже есть молодой человек.    │                  │
│ живу в Красноярске».                          │ Его зовут Владимир.                          │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ Однако, чтобы не обидеть уже немолодого       │ Спартаковцы ехали на футбол, и их болельщики │ [0.9756, 0.0244] │
│ абитуриента, девушки сказали ему, что у него  │ ехали на футбол. В метро было много людей. К │                  │
│ очень красивое и выразительное лицо, и        │ счастью, футболисты никого не потеряли. Все  │                  │
│ посоветовали ему поступать на актёрский       │ приехали на стадион вовремя.                 │                  │
│ факультет.                                    │                                              │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ Если у вашего собеседника есть домашний       │ И вы знаете, что случилось в этот вечер?!    │ [0.9984, 0.0016] │
│ любимец, он с удовольствием о нём расскажет   │ Анна тоже была на концерте. Это был также и  │                  │
│ и, может быть, покажет фотографии. хобби =    │ её любимый музыкант. Она не знала, что я     │                  │
│ увлечение. Всегда интересно говорить о хобби. │ хотел идти на концерт.                       │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ По мнению учёных, причиной экологической      │ Говоря

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ chosen_text                                   ┃ rejected_text                                ┃ logits           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Это значит, мы приезжаем к университету, вы   │ У меня есть подруга, я её очень люблю, и всё │ [0.8465, 0.1535] │
│ выходите из машины и идёте учиться. Отвозить  │ мне в ней нравится. У неё длинные густые     │                  │
│ или отвезти кого-то куда-то – это значит      │ волнистые каштановые волосы, высокий умный   │                  │
│ привезти человека в какое-то место и оставить │ лоб и вздёрнутый кверху задорный носик.      │                  │
│ там. Понятно?                                 │                                              │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ Форт № 11 «Дёнхофф» был построен в XIX веке и │ Сегодня это не просто миллионы селфи         │ [0.7655, 0.2345] │
│ хорошо сохранился. Здесь проходят экскурсии и │ туристов и моделей. Компании часто           │                  │
│ даже квесты. По территории форта спокойно     │ рекламируют и продают продукцию в            │                  │
│ гуляют кролики, которые дают посетителям себя │ «Инстаграме». Кстати, 68 процентов           │                  │
│ погладить.                                    │ пользователей в «Инстаграме» — женщины.      │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ Его пригласили на интервью в эту компанию и   │ Эта поездка помогла мне лучше узнать Россию  │ [0.168, 0.832]   │
│ взяли на работу, потому что Сергей хорошо     │ и Украину, их историю и современность. Я     │                  │
│ знал английский язык.                         │ думаю, что я поеду в Крым ещё раз.           │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ Фотография с цветущим деревом во дворе жилого │ Она сказала, что Ульяна — очень серьёзная    │ [0.8519, 0.1481] │
│ дома размещена в социальных сетях в группе «Я │ девушка и у неё уже есть молодой человек.    │                  │
│ живу в Красноярске».                          │ Его зовут Владимир.                          │                  │
└───────────────────────────────────────────────┴──────────────────────────────────────────────┴──────────────────┘

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:2717: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


In [9]:
texts = [
    "Я люблю читать книги.",
    "Сегодня хорошая погода.",
    "Мне нравится гулять в парке.",
    "Несмотря на то, что рассматриваемая проблематика обладает многослойной структурой, её интерпретация требует дополнительной методологической рефлексии.",
    "С учётом вышеизложенного представляется необходимым пересмотреть методологические основания исследования.",
    "Дискуссия о социокультурной обусловленности языковых практик требует междисциплинарного подхода."
]

scores = pairwise_reward_model.compute_rewards(texts)
for t, s in zip(texts, scores):
    print(float(s), t)

0.019353246316313744 Я люблю читать книги.
0.025055930018424988 Сегодня хорошая погода.
0.07095491886138916 Мне нравится гулять в парке.
0.23864677548408508 Несмотря на то, что рассматриваемая проблематика обладает многослойной структурой, её интерпретация требует дополнительной методологической рефлексии.
0.20983430743217468 С учётом вышеизложенного представляется необходимым пересмотреть методологические основания исследования.
0.13976982235908508 Дискуссия о социокультурной обусловленности языковых практик требует междисциплинарного подхода.


In [10]:
results = evaluate_reward_model(pairwise_reward_model, level="B")
print(results)

{'level': 'B', 'pairwise_accuracy': 0.775609756097561, 'n_pairs': 410, 'mean_margin': 0.33037057518959045, 'A_mean': 0.28014907240867615, 'B_mean': 0.8573374152183533, 'C_mean': 0.6046276092529297}


In [11]:
results["model"] = "rugpt"
results["epochs"] = 5
df_results = pd.DataFrame([results])
df_results.to_csv("reward_model_results_rugpt_5_B.csv", index=False)
df_results

,level,pairwise_accuracy,n_pairs,mean_margin,A_mean,B_mean,C_mean,model,epochs
0,B,0.77561,410,0.330371,0.280149,0.857337,0.604628,rugpt,5


## 8 epochs - A

In [ ]:
#from modeling.pairwise_reward_model import CEFRRewardModel
#from config import PairwiseRewardModelConfig
import torch
import argparse
import warnings

# changed for colab
def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--level", type=str, default="A")
    parser.add_argument("--model_name", type=str, default="ai-forever/rugpt3small_based_on_gpt2")
    parser.add_argument("--num_labels", type=int, default=1)
    parser.add_argument("--train_batch_size", type=int, default=32)
    parser.add_argument("--eval_batch_size", type=int, default=32)
    parser.add_argument("--learning_rate", type=float, default=1.41e-5)
    parser.add_argument("--num_train_epochs", type=int, default=8)
    parser.add_argument("--gradient_accumulation_steps", type=int, default=2)
    parser.add_argument("--output_dir", type=str, default="./reward_model/")
    parser.add_argument("--dataset_num_proc", type=int, default=64)
    parser.add_argument("--max_length", type=int, default=128)
    parser.add_argument("--wandb_project", type=str, default="CEFR-RewardModel")
    parser.add_argument("--wandb_exp_name", type=str, default=None)
    # changed for colab (was return parser.parse_args())
    return parser.parse_known_args()[0]


if __name__ == "__main__":

    if torch.cuda.device_count() > 1:
        warnings.warn(f"Multiple GPU Training has not been tested")
    args = parse_args()

    print("MODEL:", args.model_name)
    print("LEVEL:", args.level)
    print("EPOCHS:", args.num_train_epochs)

    pairwise_reward_model_config = PairwiseRewardModelConfig(
        model_name=args.model_name,
        num_labels=args.num_labels,
        level=args.level,
        train_batch_size=args.train_batch_size,
        eval_batch_size=args.eval_batch_size,
        learning_rate=args.learning_rate,
        num_train_epochs=args.num_train_epochs,
        gradient_accumulation_steps=args.gradient_accumulation_steps,
        output_dir=args.output_dir,
        max_length=args.max_length,
        wandb_project=args.wandb_project,
        wandb_exp_name=args.wandb_exp_name
    )

    pairwise_reward_model = CEFRRewardModel(pairwise_reward_model_config)
    pairwise_reward_model.train()

MODEL: ai-forever/rugpt3small_based_on_gpt2
LEVEL: A
EPOCHS: 8


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at ai-forever/rugpt3small_based_on_gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loading CEFR data... 


Map:   0%|          | 0/1519 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1519 [00:00<?, ? examples/s]

...

You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:2717: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss
1,0.726600
2,0.582500
3,0.547900
4,0.524200
5,0.471700
6,0.432500
7,0.494000
8,0.373100
9,0.300100
10,0.353900


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ chosen_text                                   ┃ rejected_text                                ┃ logits           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Думали ли читатели рассказов Рэя Бредбери,    │ Слово «экология» знают сейчас все люди. Как  │ [0.9744, 0.0256] │
│ что в нашей жизни будут «социальные сети» и   │ правило, его используют для характеристики   │                  │
│ мобильные телефоны, о которых он писал?       │ условий проживания: «хорошая (плохая)        │                  │
│                                               │ экология», «в нашем районе – ужасная         │                  │
│                                               │ экология». Эти слова понятны всем.           │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ В 17:45 я иду в супермаркет и покупаю         │ В 1970 году с благословения Американской     │ [1.0, 0.0]       │
│ продукты. Потом я иду домой и готовлю ужин.   │ академии религии Гэскин едет в тур по        │                  │
│ До ужина я ещё убираю квартиру. В 20:00 я     │ Америке — беседовать о духовности с          │                  │
│ ужинаю.                                       │ молодёжью. За ним, на машинах и школьных     │                  │
│                                               │ автобусах, едут его последователи.           │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ Я помню, несколько лет назад я была в банке.  │ Потратить деньги на какую-нибудь не очень    │ [0.9999, 0.0001] │
│ Там работал очень красивый мужчина. Мы        │ нужную вещь значит забыть на пару часов о    │                  │
│ познакомились, и он пригласил меня в кино.    │ проблемах в отношениях и на работе.          │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ Вы его, конечно, тоже знаете, но называете    │ Мы отдалились от природы, поэтому не ценим и │ [0.9999, 0.0001] │
│ «русский салат». Ингредиенты в «Оливье» очень │ не уважаем её. Природа отвечает нам тем же.  │                  │
│ простые. Это картофель, морковь, зелёный      │ Примером может служить изменение погоды и    │                  │
│ горошек, яйца, солёные огурцы и мясо. Иногда  │ климата на всей планете.                     │                  │
│ этот салат делают без мяса.                   │                                              │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ Этому городу уже 500 лет. В Плёсе есть        │ Г В этом сезоне нас чаще всего «лечили»      │ [0.9817, 0.0183] │
│ Дом-музей художника Исаака Ильича Левитана.   │ простуженные граждане, на остановке          │                  │
│ Этот известный художник очень любил природу,  │ ожидающие автобус. Помните, девушка          │                  │
│ поэтому он жил и работал в этом тихом зелёном │ «вылечила» от простуды стареющего рокера: —  │                  │
│ городе.                                       │ Апчхи. О, эта простуда.                      │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ На вертолёте он перелетел через Красное море, │ Гости могут принести вам хлеб и соль, это    │ [1.0, 0.0]       │
│ прилетел в Аравию, а оттуда на машине поехал  │ символический жест по немецкой традиции.     │                  │
│ в Иран.                                       │ Подарки дарят разные.                        │                  │
├───────────────────────────────────────────────┼───────

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ chosen_text                                   ┃ rejected_text                                ┃ logits           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Думали ли читатели рассказов Рэя Бредбери,    │ Слово «экология» знают сейчас все люди. Как  │ [0.9744, 0.0256] │
│ что в нашей жизни будут «социальные сети» и   │ правило, его используют для характеристики   │                  │
│ мобильные телефоны, о которых он писал?       │ условий проживания: «хорошая (плохая)        │                  │
│                                               │ экология», «в нашем районе – ужасная         │                  │
│                                               │ экология». Эти слова понятны всем.           │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ В 17:45 я иду в супермаркет и покупаю         │ В 1970 году с благословения Американской     │ [1.0, 0.0]       │
│ продукты. Потом я иду домой и готовлю ужин.   │ академии религии Гэскин едет в тур по        │                  │
│ До ужина я ещё убираю квартиру. В 20:00 я     │ Америке — беседовать о духовности с          │                  │
│ ужинаю.                                       │ молодёжью. За ним, на машинах и школьных     │                  │
│                                               │ автобусах, едут его последователи.           │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ Я помню, несколько лет назад я была в банке.  │ Потратить деньги на какую-нибудь не очень    │ [0.9999, 0.0001] │
│ Там работал очень красивый мужчина. Мы        │ нужную вещь значит забыть на пару часов о    │                  │
│ познакомились, и он пригласил меня в кино.    │ проблемах в отношениях и на работе.          │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ Вы его, конечно, тоже знаете, но называете    │ Мы отдалились от природы, поэтому не ценим и │ [0.9999, 0.0001] │
│ «русский салат». Ингредиенты в «Оливье» очень │ не уважаем её. Природа отвечает нам тем же.  │                  │
│ простые. Это картофель, морковь, зелёный      │ Примером может служить изменение погоды и    │                  │
│ горошек, яйца, солёные огурцы и мясо. Иногда  │ климата на всей планете.                     │                  │
│ этот салат делают без мяса.                   │                                              │                  │
└───────────────────────────────────────────────┴──────────────────────────────────────────────┴──────────────────┘

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:2717: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


In [ ]:
texts = [
    "Я люблю читать книги.",
    "Сегодня хорошая погода.",
    "Мне нравится гулять в парке.",
    "Несмотря на то, что рассматриваемая проблематика обладает многослойной структурой, её интерпретация требует дополнительной методологической рефлексии.",
    "С учётом вышеизложенного представляется необходимым пересмотреть методологические основания исследования.",
    "Дискуссия о социокультурной обусловленности языковых практик требует междисциплинарного подхода."
]

scores = pairwise_reward_model.compute_rewards(texts)
for t, s in zip(texts, scores):
    print(float(s), t)

0.9999927282333374 Я люблю читать книги.
0.9999852180480957 Сегодня хорошая погода.
0.9994509816169739 Мне нравится гулять в парке.
0.012728926725685596 Несмотря на то, что рассматриваемая проблематика обладает многослойной структурой, её интерпретация требует дополнительной методологической рефлексии.
0.004405633080750704 С учётом вышеизложенного представляется необходимым пересмотреть методологические основания исследования.
0.005335689056664705 Дискуссия о социокультурной обусловленности языковых практик требует междисциплинарного подхода.


In [ ]:
results = evaluate_reward_model(pairwise_reward_model, level="A")
print(results)

{'level': 'A', 'pairwise_accuracy': 0.9259259259259259, 'n_pairs': 189, 'mean_margin': 0.572459876537323, 'A_mean': 0.9822330474853516, 'B_mean': 0.4304565489292145, 'C_mean': 0.17710892856121063}


In [ ]:
results["model"] = "rugpt"
results["epochs"] = 8
df_results = pd.DataFrame([results])
df_results.to_csv("reward_model_results_rugpt_8_A.csv", index=False)
df_results

,level,pairwise_accuracy,n_pairs,mean_margin,A_mean,B_mean,C_mean,model,epochs
0,A,0.925926,189,0.57246,0.982233,0.430457,0.177109,rugpt,8


## 8 epochs - B

In [ ]:
#from modeling.pairwise_reward_model import CEFRRewardModel
#from config import PairwiseRewardModelConfig
import torch
import argparse
import warnings

# changed for colab
def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--level", type=str, default="B")
    parser.add_argument("--model_name", type=str, default="openai-community/gpt2")
    parser.add_argument("--num_labels", type=int, default=1)
    parser.add_argument("--train_batch_size", type=int, default=32)
    parser.add_argument("--eval_batch_size", type=int, default=32)
    parser.add_argument("--learning_rate", type=float, default=1.41e-5)
    parser.add_argument("--num_train_epochs", type=int, default=8)
    parser.add_argument("--gradient_accumulation_steps", type=int, default=2)
    parser.add_argument("--output_dir", type=str, default="./reward_model/")
    parser.add_argument("--dataset_num_proc", type=int, default=64)
    parser.add_argument("--max_length", type=int, default=128)
    parser.add_argument("--wandb_project", type=str, default="CEFR-RewardModel")
    parser.add_argument("--wandb_exp_name", type=str, default=None)
    # changed for colab (was return parser.parse_args())
    return parser.parse_known_args()[0]


if __name__ == "__main__":

    if torch.cuda.device_count() > 1:
        warnings.warn(f"Multiple GPU Training has not been tested")
    args = parse_args()
    pairwise_reward_model_config = PairwiseRewardModelConfig(
        model_name=args.model_name,
        num_labels=args.num_labels,
        level=args.level,
        train_batch_size=args.train_batch_size,
        eval_batch_size=args.eval_batch_size,
        learning_rate=args.learning_rate,
        num_train_epochs=args.num_train_epochs,
        gradient_accumulation_steps=args.gradient_accumulation_steps,
        output_dir=args.output_dir,
        max_length=args.max_length,
        wandb_project=args.wandb_project,
        wandb_exp_name=args.wandb_exp_name
    )

    pairwise_reward_model = CEFRRewardModel(pairwise_reward_model_config)
    pairwise_reward_model.train()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at openai-community/gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

...

You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:2717: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss
1,0.723600
2,0.727600
3,0.713300
4,0.678300
5,0.675200
6,0.665600
7,0.679900
8,0.692900


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ chosen_text                                   ┃ rejected_text                                ┃ logits           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Фотография с цветущим деревом во дворе жилого │ Она сказала, что Ульяна — очень серьёзная    │ [0.5677, 0.4323] │
│ дома размещена в социальных сетях в группе «Я │ девушка и у неё уже есть молодой человек.    │                  │
│ живу в Красноярске».                          │ Его зовут Владимир.                          │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ Когда они пришли на следующий день в этот     │ После бани хорошо поплавать в холодном       │ [0.5379, 0.4621] │
│ парк, они не увидели Перельмана. Перельман не │ бассейне или принять холодный душ. И, как    │                  │
│ пришёл.                                       │ говорят в России, «С лёгким паром!».         │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ Теперь всё изменилось: в социальных сетях     │ В римской бане тёплый пол, а горячий идёт из │ [0.427, 0.573]   │
│ люди заводят отношения с теми, кого никогда   │ стен, температура там градусов 70. Римские   │                  │
│ не видели.                                    │ бани были очень красивые.                    │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ После Москвы он является вторым по величине   │ Погода была отличная: светило солнце, было   │ [0.5068, 0.4932] │
│ городом России, крупным промышленным, научным │ тепло, но не жарко. Я прекрасно провёл время │                  │
│ и историко-культурным центром.                │ с друзьями в Суздале!                        │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ С того времени несёт она свои воды в Енисей.  │ Именно за счёт него происходит рост,         │ [0.4408, 0.5592] │
│ А камень, который бросил Байкал, и сейчас     │ развитие и деление каждой клетки.            │                  │
│ стоит на том же месте.                        │ Контролируются эти процессы гормонами.       │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ Травмы можно получить даже после таких        │ А теперь к вопросу о толерантности — вот     │ [0.5433, 0.4567] │
│ простых и лёгких физических упражнений, как   │ каков был мой опыт, опыт легкомысленно       │                  │
│ бег или езда на велосипеде.                   │ миролюбивого москвича.                       │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ А баня — это для меня праздник! Я люблю и     │ В результате прогресса медицины и кризиса    │ [0.4403, 0.5597] │
│ финскую сауну, и турецкую баню. Но больше     │ пенсионной системы люди будут работать в     │                  │
│ всего мне нравится русская баня.              │ старости, а не жить на пенсию.               │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ На этих чтениях выступают с докладами и       │ Но сторонники ничего не говорят, они просто  │ [0.5927, 0.4073] │
│ научными сообщениями очень известные, а также │ играют в компьютерные игры, потому что им    │                  │
│ молодые учёные.                               │ это интересно.                               │                  │
├───────────────────────────────────────────────┼───────

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ chosen_text                                   ┃ rejected_text                                ┃ logits           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Фотография с цветущим деревом во дворе жилого │ Она сказала, что Ульяна — очень серьёзная    │ [0.5677, 0.4323] │
│ дома размещена в социальных сетях в группе «Я │ девушка и у неё уже есть молодой человек.    │                  │
│ живу в Красноярске».                          │ Его зовут Владимир.                          │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ Когда они пришли на следующий день в этот     │ После бани хорошо поплавать в холодном       │ [0.5379, 0.4621] │
│ парк, они не увидели Перельмана. Перельман не │ бассейне или принять холодный душ. И, как    │                  │
│ пришёл.                                       │ говорят в России, «С лёгким паром!».         │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ Теперь всё изменилось: в социальных сетях     │ В римской бане тёплый пол, а горячий идёт из │ [0.427, 0.573]   │
│ люди заводят отношения с теми, кого никогда   │ стен, температура там градусов 70. Римские   │                  │
│ не видели.                                    │ бани были очень красивые.                    │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ После Москвы он является вторым по величине   │ Погода была отличная: светило солнце, было   │ [0.5068, 0.4932] │
│ городом России, крупным промышленным, научным │ тепло, но не жарко. Я прекрасно провёл время │                  │
│ и историко-культурным центром.                │ с друзьями в Суздале!                        │                  │
└───────────────────────────────────────────────┴──────────────────────────────────────────────┴──────────────────┘

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:2717: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


In [ ]:
texts = [
    "Я люблю читать книги.",
    "Сегодня хорошая погода.",
    "Мне нравится гулять в парке.",
    "Несмотря на то, что рассматриваемая проблематика обладает многослойной структурой, её интерпретация требует дополнительной методологической рефлексии.",
    "С учётом вышеизложенного представляется необходимым пересмотреть методологические основания исследования.",
    "Дискуссия о социокультурной обусловленности языковых практик требует междисциплинарного подхода."
]

scores = pairwise_reward_model.compute_rewards(texts)
for t, s in zip(texts, scores):
    print(float(s), t)

0.27769985795021057 Я люблю читать книги.
0.2963213622570038 Сегодня хорошая погода.
0.2763323485851288 Мне нравится гулять в парке.
0.3064793050289154 Несмотря на то, что рассматриваемая проблематика обладает многослойной структурой, её интерпретация требует дополнительной методологической рефлексии.
0.3084152638912201 С учётом вышеизложенного представляется необходимым пересмотреть методологические основания исследования.
0.30679914355278015 Дискуссия о социокультурной обусловленности языковых практик требует междисциплинарного подхода.


In [ ]:
results = evaluate_reward_model(pairwise_reward_model, level="B")
print(results)

{'level': 'B', 'pairwise_accuracy': 0.5536585365853659, 'n_pairs': 410, 'mean_margin': 0.004355233162641525, 'A_mean': 0.29246601462364197, 'B_mean': 0.2985416352748871, 'C_mean': 0.2984464764595032}


In [ ]:
results["model"] = "gpt2"
results["epochs"] = 8
df_results = pd.DataFrame([results])
df_results.to_csv("reward_model_results_gpt2_8_B.csv", index=False)
df_results

,level,pairwise_accuracy,n_pairs,mean_margin,A_mean,B_mean,C_mean,model,epochs
0,B,0.553659,410,0.004355,0.292466,0.298542,0.298446,gpt2,8


## 8 epochs - C

In [ ]:
#from modeling.pairwise_reward_model import CEFRRewardModel
#from config import PairwiseRewardModelConfig
import torch
import argparse
import warnings

# changed for colab
def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--level", type=str, default="C")
    parser.add_argument("--model_name", type=str, default="openai-community/gpt2")
    parser.add_argument("--num_labels", type=int, default=1)
    parser.add_argument("--train_batch_size", type=int, default=32)
    parser.add_argument("--eval_batch_size", type=int, default=32)
    parser.add_argument("--learning_rate", type=float, default=1.41e-5)
    parser.add_argument("--num_train_epochs", type=int, default=8)
    parser.add_argument("--gradient_accumulation_steps", type=int, default=2)
    parser.add_argument("--output_dir", type=str, default="./reward_model/")
    parser.add_argument("--dataset_num_proc", type=int, default=64)
    parser.add_argument("--max_length", type=int, default=128)
    parser.add_argument("--wandb_project", type=str, default="CEFR-RewardModel")
    parser.add_argument("--wandb_exp_name", type=str, default=None)
    # changed for colab (was return parser.parse_args())
    return parser.parse_known_args()[0]


if __name__ == "__main__":

    if torch.cuda.device_count() > 1:
        warnings.warn(f"Multiple GPU Training has not been tested")
    args = parse_args()
    pairwise_reward_model_config = PairwiseRewardModelConfig(
        model_name=args.model_name,
        num_labels=args.num_labels,
        level=args.level,
        train_batch_size=args.train_batch_size,
        eval_batch_size=args.eval_batch_size,
        learning_rate=args.learning_rate,
        num_train_epochs=args.num_train_epochs,
        gradient_accumulation_steps=args.gradient_accumulation_steps,
        output_dir=args.output_dir,
        max_length=args.max_length,
        wandb_project=args.wandb_project,
        wandb_exp_name=args.wandb_exp_name
    )

    pairwise_reward_model = CEFRRewardModel(pairwise_reward_model_config)
    pairwise_reward_model.train()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at openai-community/gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loading CEFR data... 


Map:   0%|          | 0/945 [00:00<?, ? examples/s]

Filter:   0%|          | 0/945 [00:00<?, ? examples/s]

Map:   0%|          | 0/118 [00:00<?, ? examples/s]

...

You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:2717: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss
1,0.567100
2,0.369800
3,0.336100
4,0.361300
5,0.402200
6,0.336500
7,0.357900
8,0.439100


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ chosen_text                                   ┃ rejected_text                                ┃ logits           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Логично было бы делать ряды этих товаров      │ А баня — это для меня праздник! Я люблю и    │ [0.4469, 0.5531] │
│ ближе ко входу, чтобы покупатели не тратили   │ финскую сауну, и турецкую баню. Но больше    │                  │
│ время, но происходит наоборот.                │ всего мне нравится русская баня.             │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ Именно за счёт него происходит рост, развитие │ Несмотря на то, что стены из белого камня    │ [0.692, 0.308]   │
│ и деление каждой клетки. Контролируются эти   │ были непрочными, название «Белокаменная»     │                  │
│ процессы гормонами.                           │ живёт и сегодня.                             │                  │
└───────────────────────────────────────────────┴──────────────────────────────────────────────┴──────────────────┘

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ chosen_text                                   ┃ rejected_text                                ┃ logits           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Логично было бы делать ряды этих товаров      │ А баня — это для меня праздник! Я люблю и    │ [0.4469, 0.5531] │
│ ближе ко входу, чтобы покупатели не тратили   │ финскую сауну, и турецкую баню. Но больше    │                  │
│ время, но происходит наоборот.                │ всего мне нравится русская баня.             │                  │
├───────────────────────────────────────────────┼──────────────────────────────────────────────┼──────────────────┤
│ Именно за счёт него происходит рост, развитие │ Несмотря на то, что стены из белого камня    │ [0.692, 0.308]   │
│ и деление каждой клетки. Контролируются эти   │ были непрочными, название «Белокаменная»     │                  │
│ процессы гормонами.                           │ живёт и сегодня.                             │                  │
└───────────────────────────────────────────────┴──────────────────────────────────────────────┴──────────────────┘

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:2717: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


In [ ]:
texts = [
    "Я люблю читать книги.",
    "Сегодня хорошая погода.",
    "Мне нравится гулять в парке.",
    "Несмотря на то, что рассматриваемая проблематика обладает многослойной структурой, её интерпретация требует дополнительной методологической рефлексии.",
    "С учётом вышеизложенного представляется необходимым пересмотреть методологические основания исследования.",
    "Дискуссия о социокультурной обусловленности языковых практик требует междисциплинарного подхода."
]

scores = pairwise_reward_model.compute_rewards(texts)
for t, s in zip(texts, scores):
    print(float(s), t)

0.5724639296531677 Я люблю читать книги.
0.6114470958709717 Сегодня хорошая погода.
0.5905376076698303 Мне нравится гулять в парке.
0.7204639315605164 Несмотря на то, что рассматриваемая проблематика обладает многослойной структурой, её интерпретация требует дополнительной методологической рефлексии.
0.7067111134529114 С учётом вышеизложенного представляется необходимым пересмотреть методологические основания исследования.
0.743971586227417 Дискуссия о социокультурной обусловленности языковых практик требует междисциплинарного подхода.


In [ ]:
results = evaluate_reward_model(pairwise_reward_model, level="C")
print(results)

{'level': 'C', 'pairwise_accuracy': 0.5798319327731093, 'n_pairs': 119, 'mean_margin': -8.164533937815577e-05, 'A_mean': 0.6663765907287598, 'B_mean': 0.6791114211082458, 'C_mean': 0.6785526871681213}


In [ ]:
results["model"] = "gpt2"
results["epochs"] = 8
df_results = pd.DataFrame([results])
df_results.to_csv("reward_model_results_gpt2_8_C.csv", index=False)
df_results

,level,pairwise_accuracy,n_pairs,mean_margin,A_mean,B_mean,C_mean,model,epochs
0,C,0.579832,119,-0.000082,0.666377,0.679111,0.678553,gpt2,8
